# Step 5: Model Training (Day5)

In [21]:
# Load and prepare data
import ast
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('D:/Python/NLP/CK/final_data.csv')
df['tokens'] = df['tokens'].apply(ast.literal_eval)
df['labels'] = df['labels'].apply(ast.literal_eval)
df = df[df['tokens'].apply(len) == df['labels'].apply(len)].reset_index(drop=True)
print(f"Loaded {len(df)} sentences")

all_labels = set()
for labels in df['labels']:
    all_labels.update(labels)
label_list = sorted(list(all_labels))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(f"Labels: {len(label_list)}")

Loaded 3113 sentences
Labels: 32


In [22]:
# Load tokenizer and model
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForTokenClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)
with open('D:/Python/NLP/CK/label_mappings.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f)
print(f"Model ready: {model.num_parameters():,} params")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4089.53it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model ready: 66,387,488 params


In [23]:
# Prepare data function
def prepare_data(examples, tokenizer, max_length=128):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )
    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized_inputs.word_ids(i)
        label_ids = []
        prev = None
        for w in word_ids:
            if w is None:
                label_ids.append(-100)
            elif w != prev:
                label_ids.append(label2id.get(label[w], 0))
            else:
                label_ids.append(-100)
            prev = w
        labels.append(label_ids)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

# Split data
train_df, temp = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp, test_size=0.5, random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 2490, Val: 311, Test: 312


In [24]:
# Tokenize
print("Tokenizing...")
train_ds = Dataset.from_pandas(train_df[['tokens', 'labels']]).map(lambda x: prepare_data(x, tokenizer), batched=True, remove_columns=['tokens', 'labels'])
val_ds = Dataset.from_pandas(val_df[['tokens', 'labels']]).map(lambda x: prepare_data(x, tokenizer), batched=True, remove_columns=['tokens', 'labels'])
test_ds = Dataset.from_pandas(test_df[['tokens', 'labels']]).map(lambda x: prepare_data(x, tokenizer), batched=True, remove_columns=['tokens', 'labels'])
print(f"Done - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Tokenizing...


Map: 100%|██████████| 312/312 [00:00<00:00, 6388.73 examples/s]

Done - Train: 2490, Val: 311, Test: 312


In [25]:
# Metrics
def compute_metrics(eval_pred):
    from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
    predictions = np.argmax(eval_pred.predictions, axis=2)
    labels = eval_pred.label_ids
    true_pred, true_label = [], []
    for pred, label in zip(predictions, labels):
        tp, tl = [], []
        for p, l in zip(pred, label):
            if l != -100:
                tp.append(id2label[int(p)])
                tl.append(id2label[int(l)])
        if tp:
            true_pred.append(tp)
            true_label.append(tl)
    print("\n" + classification_report(true_label, true_pred, digits=4))
    return {"f1": f1_score(true_label, true_pred), "precision": precision_score(true_label, true_pred), "recall": recall_score(true_label, true_pred)}

In [26]:
# Training
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="D:/Python/NLP/CK/ner_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=50,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    seed=42,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=5))

print("Training... LR=2e-5, Epochs=15, Batch=16")
trainer.train()

Training... LR=2e-5, Epochs=15, Batch=16


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,No log,0.246947,0.620370,0.616375,0.624418
2,No log,0.171394,0.766557,0.772727,0.760485
3,No log,0.149149,0.789748,0.804642,0.775396
4,0.415499,0.139597,0.800191,0.821394,0.780056
5,0.415499,0.138434,0.811374,0.825458,0.797763
6,0.415499,0.136771,0.831181,0.853635,0.809879
7,0.084092,0.131991,0.824536,0.841748,0.808015
8,0.084092,0.138731,0.834204,0.850775,0.818267
9,0.084092,0.137178,0.834443,0.852284,0.817335
10,0.046908,0.142859,0.819672,0.823917,0.815471



              precision    recall  f1-score   support

    CARDINAL     0.7778    0.7975    0.7875        79
        DATE     0.6433    0.8080    0.7163       125
         FAC     0.0000    0.0000    0.0000         4
         GPE     0.6000    0.9079    0.7225       152
    LANGUAGE     0.0000    0.0000    0.0000         1
         LAW     0.0000    0.0000    0.0000         1
         LOC     0.5000    0.0435    0.0800        46
       MONEY     0.8824    0.8571    0.8696        35
        NORP     0.7292    0.3365    0.4605       104
     ORDINAL     0.0000    0.0000    0.0000        11
         ORG     0.5477    0.5789    0.5629       228
      PERSON     0.5788    0.6145    0.5961       275
     PRODUCT     0.0000    0.0000    0.0000         7
        TIME     0.0000    0.0000    0.0000         1
 WORK_OF_ART     0.0000    0.0000    0.0000         4

   micro avg     0.6164    0.6244    0.6204      1073
   macro avg     0.3506    0.3296    0.3197      1073
weighted avg     0.6028  

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.52it/s]



              precision    recall  f1-score   support

    CARDINAL     0.9041    0.8354    0.8684        79
        DATE     0.8633    0.9600    0.9091       125
         FAC     1.0000    1.0000    1.0000         4
         GPE     0.8563    0.9408    0.8966       152
    LANGUAGE     0.0000    0.0000    0.0000         1
         LAW     0.0000    0.0000    0.0000         1
         LOC     0.3333    0.2826    0.3059        46
       MONEY     0.7778    1.0000    0.8750        35
        NORP     0.7595    0.5769    0.6557       104
     ORDINAL     1.0000    0.9091    0.9524        11
         ORG     0.6281    0.6667    0.6468       228
      PERSON     0.8256    0.7745    0.7992       275
     PRODUCT     0.0000    0.0000    0.0000         7
        TIME     0.0000    0.0000    0.0000         1
 WORK_OF_ART     0.0000    0.0000    0.0000         4

   micro avg     0.7727    0.7605    0.7666      1073
   macro avg     0.5299    0.5297    0.5273      1073
weighted avg     0.7607  

...


              precision    recall  f1-score   support

    CARDINAL     0.9863    0.9114    0.9474        79
        DATE     0.9444    0.9520    0.9482       125
         FAC     1.0000    1.0000    1.0000         4
         GPE     0.9551    0.9803    0.9675       152
    LANGUAGE     0.0000    0.0000    0.0000         1
         LAW     0.0000    0.0000    0.0000         1
         LOC     0.7000    0.6087    0.6512        46
       MONEY     0.8293    0.9714    0.8947        35
        NORP     0.7368    0.6731    0.7035       104
     ORDINAL     1.0000    1.0000    1.0000        11
         ORG     0.7923    0.7193    0.7540       228
      PERSON     0.8500    0.8036    0.8262       275
     PRODUCT     0.3636    0.5714    0.4444         7
        TIME     1.0000    1.0000    1.0000         1
 WORK_OF_ART     1.0000    0.7500    0.8571         4

   micro avg     0.8552    0.8201    0.8373      1073
   macro avg     0.7439    0.7294    0.7330      1073
weighted avg     0.8536  

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.23it/s]



              precision    recall  f1-score   support

    CARDINAL     0.9863    0.9114    0.9474        79
        DATE     0.9444    0.9520    0.9482       125
         FAC     1.0000    1.0000    1.0000         4
         GPE     0.9554    0.9868    0.9709       152
    LANGUAGE     0.0000    0.0000    0.0000         1
         LAW     0.0000    0.0000    0.0000         1
         LOC     0.7632    0.6304    0.6905        46
       MONEY     0.8293    0.9714    0.8947        35
        NORP     0.6970    0.6635    0.6798       104
     ORDINAL     1.0000    1.0000    1.0000        11
         ORG     0.7826    0.7105    0.7448       228
      PERSON     0.8359    0.7964    0.8156       275
     PRODUCT     0.3636    0.5714    0.4444         7
        TIME     1.0000    1.0000    1.0000         1
 WORK_OF_ART     1.0000    0.7500    0.8571         4

   micro avg     0.8482    0.8173    0.8325      1073
   macro avg     0.7438    0.7296    0.7329      1073
weighted avg     0.8468  

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.28it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2340, training_loss=0.12612024168682914, metrics={'train_runtime': 9009.0094, 'train_samples_per_second': 4.146, 'train_steps_per_second': 0.26, 'total_flos': 1220635079884800.0, 'train_loss': 0.12612024168682914, 'epoch': 15.0})

In [27]:
# Save and evaluate
model_path = "D:/Python/NLP/CK/ner_model_final"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Saved: {model_path}")

results = trainer.predict(test_ds)
print(f"\nTest Results:")
print(f"  F1: {results.metrics['test_f1']:.4f}")
print(f"  Precision: {results.metrics['test_precision']:.4f}")
print(f"  Recall: {results.metrics['test_recall']:.4f}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.67it/s]


Saved: D:/Python/NLP/CK/ner_model_final



              precision    recall  f1-score   support

    CARDINAL     0.9032    0.9180    0.9106        61
        DATE     0.9441    0.9783    0.9609       138
         FAC     1.0000    0.8333    0.9091         6
         GPE     0.8921    0.9688    0.9288       128
    LANGUAGE     0.0000    0.0000    0.0000         1
         LAW     0.0000    0.0000    0.0000         1
         LOC     0.6833    0.6721    0.6777        61
       MONEY     0.9444    1.0000    0.9714        17
        NORP     0.7982    0.7647    0.7811       119
     ORDINAL     1.0000    1.0000    1.0000        17
         ORG     0.8119    0.7629    0.7867       232
      PERSON     0.8863    0.8370    0.8610       270
     PRODUCT     0.9167    0.9167    0.9167        12
    QUANTITY     0.3333    0.2500    0.2857         8
        TIME     0.0000    0.0000    0.0000         1
 WORK_OF_ART     0.7500    0.6000    0.6667        10

   micro avg     0.8590    0.8392    0.8490      1082
   macro avg     0.6790  

In [31]:
# Test predictions
from transformers import AutoModelForTokenClassification
import torch

model = AutoModelForTokenClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

def predict(text):
    words = text.split()  # 👈 quan trọng

    inputs = tokenizer(words, is_split_into_words=True,
                       return_tensors="pt", truncation=True, max_length=128)

    with torch.no_grad():
        preds = torch.argmax(model(**inputs).logits, dim=2)[0]

    tokens = inputs.tokens()
    word_ids = inputs.word_ids()

    result = []
    prev = None
    for t, p, w in zip(tokens, preds.tolist(), word_ids):
        if w != prev and w is not None:
            result.append((words[w], id2label.get(p, 'O')))
        prev = w

    return result

tests = [
    "Joe Biden met Donald Trump in Washington D.C. on Monday",
    "Apple Inc. announced $1 billion investment",
    "The meeting is at the White House on January 15"
]

print("\nPredictions:")
for text in tests:
    preds = predict(text)
    print(f"\n{text}")
    for t, l in preds:
        print(f"{t} -> {l}")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 6346.52it/s]



Predictions:

Joe Biden met Donald Trump in Washington D.C. on Monday
Joe -> B-PERSON
Biden -> I-PERSON
met -> B-ORG
Donald -> B-PERSON
Trump -> I-PERSON
in -> O
Washington -> B-GPE
D.C. -> I-GPE
on -> O
Monday -> B-DATE

Apple Inc. announced $1 billion investment
Apple -> B-ORG
Inc. -> I-ORG
announced -> O
$1 -> B-MONEY
billion -> I-MONEY
investment -> O

The meeting is at the White House on January 15
The -> O
meeting -> O
is -> O
at -> O
the -> O
White -> B-FAC
House -> I-FAC
on -> O
January -> B-DATE
15 -> I-DATE
